# Johansen CO2 Sequestration — Surveillance Well Selection

This notebook uses a **multi-criteria scoring framework** to give a data-driven recommendation
for which single real drilled well should serve as the primary CO2 surveillance well
for the Johansen formation CO2 storage project.

**Candidate wells:** `31_01_01`, `31_1-3_S`, `31_2-5`, `31_4-3`, `31_05_02`, `31_07_01`

**Excluded (not candidates):**
- `31_05_07` — the CO2 injector
- `SBoundary_test_well` — a proposed (not drilled) fictional sentinel

**Five evaluation criteria:**
1. Earliest CO2 detection (sensitivity)
2. Pressure signal strength (detectability)
3. Layer of first detection (buoyancy sensitivity)
4. Consistency across injection rates (robustness)
5. Structural position / depth (updip proximity)

In [ ]:
# === PARAMETERS — change only here ===
from pathlib import Path

DATA_ROOT           = Path("/Users/apple/Desktop/study/programming/Matlab/Plugins/MRST-2026a/core/examples/data/Johansen/well_csvs")
INJECTOR_WELL       = "31_05_07"
EXCLUDE_WELLS       = ["31_05_07", "SBoundary_test_well"]
S_DETECTION_THRESHOLD = 0.001    # CO2 saturation to count as 'detected'
P_FRACTURE_BAR      = 380        # caprock fracture pressure [bar]
LAYERS              = [6, 7, 8, 9, 10]
CANDIDATES          = ["31_01_01", "31_1-3_S", "31_2-5", "31_4-3", "31_05_02", "31_07_01"]

WEIGHTS = {
    "detection_speed":     0.30,
    "pressure_signal":     0.25,
    "buoyancy_layer":      0.20,
    "consistency":         0.15,
    "structural_position": 0.10,
}
print("Parameters loaded.")

## Section 0 — Data Ingestion

Load all simulation runs. Extract per-well depth from `Depth_m_Lk` columns. Print a depth-ranked table.

In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from scipy.interpolate import interp1d

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'figure.dpi': 130, 'font.family': 'DejaVu Sans',
                            'axes.spines.top': False, 'axes.spines.right': False})

SAT_COLS = [f'S_CO2_L{k}' for k in LAYERS]
PRE_COLS = [f'P_bar_L{k}' for k in LAYERS]
DEP_COLS = [f'Depth_m_L{k}' for k in LAYERS]

def parse_summary(txt_path):
    meta = {'Q_Mt_yr': None, 'T_inj_yr': None, 'total_injected_Mt': None, 'peak_BHP_bar': None}
    try:
        text = txt_path.read_text()
        for pat, key in [
            (r'Injector\s+([\d.]+)\s+Mt/yr', 'Q_Mt_yr'),
            (r'Injection end year\s*:\s*([\d.]+)', 'T_inj_yr'),
            (r'Total CO2 injected\s*:\s*([\d.]+)', 'total_injected_Mt'),
            (r'Peak injector BHP\s*:\s*([\d.]+)', 'peak_BHP_bar'),
        ]:
            m = re.search(pat, text)
            if m: meta[key] = float(m.group(1))
    except Exception as e:
        print(f'  [WARN] {txt_path}: {e}')
    return meta

def load_run(folder_path):
    folder_path = Path(folder_path)
    meta = parse_summary(folder_path / 'simulation_summary.txt')
    meta['folder'] = str(folder_path)
    meta['timestamp'] = folder_path.name
    wells = {}
    for csv_file in sorted(folder_path.glob('*.csv')):
        try:
            wells[csv_file.stem] = pd.read_csv(csv_file)
        except Exception as e:
            print(f'  [WARN] {csv_file}: {e}')
    return {'meta': meta, 'wells': wells}

def load_all_runs(subdir):
    runs = []
    for folder in sorted((DATA_ROOT / subdir).iterdir()):
        if folder.is_dir() and not folder.name.startswith('.'):
            runs.append(load_run(folder))
    return runs

q_runs = load_all_runs('q_vary')
t_runs = load_all_runs('t_vary')
ALL_RUNS = q_runs + t_runs
print(f'Loaded {len(q_runs)} q_vary runs, {len(t_runs)} t_vary runs')

In [ ]:
# Summary table
rows = []
for tag, runs in [('q_vary', q_runs), ('t_vary', t_runs)]:
    for r in runs:
        m = r['meta']
        rows.append({'Dataset': tag, 'Timestamp': m['timestamp'],
                     'Q (Mt/yr)': m['Q_Mt_yr'], 'T_inj (yr)': m['T_inj_yr'],
                     'Total CO2 (Mt)': m['total_injected_Mt'], 'Peak BHP (bar)': m['peak_BHP_bar'],
                     'Wells loaded': len(r['wells'])})
df_summary = pd.DataFrame(rows).sort_values(['Dataset','Q (Mt/yr)','T_inj (yr)']).reset_index(drop=True)
display(df_summary)

In [ ]:
# Extract average depth per candidate well from first run with data
well_depths = {}
for w in CANDIDATES:
    depth_vals = []
    for r in ALL_RUNS:
        df = r['wells'].get(w)
        if df is not None and not df.empty:
            cols = [c for c in DEP_COLS if c in df.columns]
            if cols:
                depth_vals.append(df[cols].iloc[0].mean())
    well_depths[w] = float(np.mean(depth_vals)) if depth_vals else np.nan

depth_df = pd.DataFrame({'Well': list(well_depths.keys()),
                         'Mean Depth (m)': list(well_depths.values())})
depth_df = depth_df.sort_values('Mean Depth (m)').reset_index(drop=True)
depth_df['Depth Rank (1=shallowest)'] = range(1, len(depth_df)+1)
print('Depth-ranked candidate wells (shallowest = most updip):')  
display(depth_df)

## Section 1 — Physical Context: Why Well Position Matters

**Geological setting:** The Johansen formation dips northward at a gentle angle. Supercritical CO2,
being buoyant relative to formation brine (~686 kg/m³ vs ~1000 kg/m³), migrates updip
toward shallower, structurally higher positions along the northern flank.

**Surveillance strategy:** An ideal surveillance well should:
1. **Detect CO2 early** — positioned along the migration pathway, not behind the plume.
2. **Show a strong pressure anomaly** — pressure fronts travel faster than the CO2 plume itself,
   enabling earlier operational response.
3. **Catch buoyancy-driven CO2 in upper layers** — layer L6 (shallowest) is the first to receive
   gravitationally migrating CO2.
4. **Be reliable across a range of injection rates** — operational decisions require a well that
   is useful whether injecting at 0.875 Mt/yr or higher rates.
5. **Be structurally updip** — closer to the CO2 migration terminus means earlier interception.

**Five quantitative criteria** are scored and combined into a weighted composite to produce
a single, defensible recommendation.

## Section 2 — Criterion 1: Earliest CO2 Detection (Sensitivity)

**Physical rationale:** A surveillance well must detect CO2 as early as possible to
allow operational response before a containment breach.

For each well and run, `T_detect` = first timestep where max CO2 saturation (L6–L10)
exceeds `S_DETECTION_THRESHOLD = 0.001`. If never detected → `T_detect = 1000` yr.

Lower `T_detect_mean` = better sentinel.

In [ ]:
def s_max_series(df):
    if df is None or df.empty: return None, None
    cols = [c for c in SAT_COLS if c in df.columns]
    if not cols: return None, None
    return df['Time_yr'], df[cols].max(axis=1)

def find_T_detect(df, threshold=None):
    if threshold is None: threshold = S_DETECTION_THRESHOLD
    if df is None or df.empty or 'Time_yr' not in df.columns: return 1000.0
    t, s = s_max_series(df)
    if s is None: return 1000.0
    breach = df.loc[s > threshold, 'Time_yr']
    return float(breach.iloc[0]) if len(breach) > 0 else 1000.0

# Build T_detect matrix: wells x runs
run_labels = [f"Q={r['meta']['Q_Mt_yr']:.2f}\nT={int(r['meta']['T_inj_yr'])}yr" for r in ALL_RUNS]
tdet_matrix = {}
for w in CANDIDATES:
    tdet_matrix[w] = [find_T_detect(r['wells'].get(w)) for r in ALL_RUNS]

tdet_df = pd.DataFrame(tdet_matrix, index=run_labels).T
tdet_df.index.name = 'Well'
tdet_df['T_detect_mean'] = tdet_df.mean(axis=1)
tdet_df['T_detect_min']  = tdet_df.drop(columns='T_detect_mean').min(axis=1)
print('T_detect matrix (years):')  
display(tdet_df.round(0))

In [ ]:
# Score_1: normalize T_detect_mean ascending (lower = better = higher score)
tmean = tdet_df['T_detect_mean']
score1 = 1 - (tmean - tmean.min()) / (tmean.max() - tmean.min() + 1e-9)
score1 = score1.rename('Score_1_detection_speed')

# Heatmap of T_detect across all runs
run_cols = [c for c in tdet_df.columns if c not in ['T_detect_mean','T_detect_min']]
heat_data = tdet_df[run_cols].values.astype(float)

fig, ax = plt.subplots(figsize=(max(10, len(run_cols)*0.9), 4.5))
cmap = matplotlib.colors.LinearSegmentedColormap.from_list('rg', ['#2ECC71','#F39C12','#E74C3C'])
im = ax.imshow(heat_data, aspect='auto', cmap=cmap, vmin=0, vmax=1000)
ax.set_xticks(range(len(run_cols)))
ax.set_xticklabels(run_cols, fontsize=7, rotation=45, ha='right')
ax.set_yticks(range(len(CANDIDATES)))
ax.set_yticklabels(tdet_df.index.tolist(), fontsize=9)
for i in range(heat_data.shape[0]):
    for j in range(heat_data.shape[1]):
        v = heat_data[i, j]
        txt = f'{v:.0f}' if v < 1000 else '>999'
        ax.text(j, i, txt, ha='center', va='center', fontsize=7,
                color='white' if v > 400 else 'black')
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('T_detect (years)', fontsize=9)
ax.set_title('Criterion 1 — T_detect Heatmap (years to CO2 detection; green = early)', fontsize=12, fontweight='bold')
ax.set_xlabel('Run (Q, T_inj)', fontsize=10); ax.set_ylabel('Candidate Well', fontsize=10)
plt.tight_layout(); plt.show()
print(f'Score_1 (detection speed, 0-1, higher=better):\n{score1.round(3).to_string()}')

## Section 3 — Criterion 2: Pressure Signal Strength (Detectability)

**Physical rationale:** Pressure fronts propagate much faster than the CO2 plume itself
(essentially instantaneous on reservoir timescales). A well showing a strong pressure anomaly
above baseline is operationally useful long before CO2 physically arrives.

For each well, `ΔP_max` = max pressure rise above baseline (t=1 yr) across all layers
and all timesteps during injection. We average this over all q_vary runs.

In [ ]:
def compute_delta_P(df):
    if df is None or df.empty or 'Time_yr' not in df.columns: return 0.0
    cols = [c for c in PRE_COLS if c in df.columns]
    if not cols: return 0.0
    baseline = df[cols].iloc[0]  # t=first timestep
    dP = (df[cols] - baseline).max().max()
    return float(dP) if not np.isnan(dP) else 0.0

dp_by_run = {w: [] for w in CANDIDATES}
dp_labels  = []
for r in q_runs:
    lbl = f"Q={r['meta']['Q_Mt_yr']:.2f}"
    dp_labels.append(lbl)
    for w in CANDIDATES:
        dp_by_run[w].append(compute_delta_P(r['wells'].get(w)))

dp_mean = {w: float(np.mean(dp_by_run[w])) for w in CANDIDATES}
score2_raw = pd.Series(dp_mean)
score2 = (score2_raw - score2_raw.min()) / (score2_raw.max() - score2_raw.min() + 1e-9)
score2 = score2.rename('Score_2_pressure_signal')

# Grouped bar + scatter
x = np.arange(len(CANDIDATES))
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.tab10(np.linspace(0, 0.7, len(CANDIDATES)))
bars = ax.bar(x, [dp_mean[w] for w in CANDIDATES], color=colors, alpha=0.75, edgecolor='white', zorder=2)
for i, w in enumerate(CANDIDATES):
    for val in dp_by_run[w]:
        ax.scatter(i, val, color='black', s=25, zorder=4, alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(CANDIDATES, fontsize=9, rotation=20, ha='right')
ax.set_ylabel('Max Pressure Rise ΔP (bar)', fontsize=11)
ax.set_title('Criterion 2 — Max Pressure Anomaly Above Baseline\n(dots = individual q_vary runs)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y', zorder=1)
plt.tight_layout(); plt.show()
print(f'Score_2 (pressure signal, 0-1, higher=better):\n{score2.round(3).to_string()}')

## Section 4 — Criterion 3: Layer of First Detection (Buoyancy Sensitivity)

**Physical rationale:** CO2 rises buoyantly into shallower layers first
(L6 = shallowest). A well that detects CO2 in L6 or L7 is intercepting the gravitationally
driven plume tip — the most physically meaningful signal for early escape risk.

For each run where CO2 is detected, we record which layer first exceeds `S_DETECTION_THRESHOLD`.
Layer scoring: L6=5, L7=4, L8=3, L9=2, L10=1. Higher score = better buoyancy sensitivity.

In [ ]:
LAYER_SCORE_MAP = {6:5, 7:4, 8:3, 9:2, 10:1}

def first_detection_layer(df):
    """Return which layer k first crosses threshold, or None."""
    if df is None or df.empty or 'Time_yr' not in df.columns: return None
    for _, row in df.iterrows():
        for k in LAYERS:
            col = f'S_CO2_L{k}'
            if col in row and row[col] > S_DETECTION_THRESHOLD:
                return k
    return None

layer_tallies = {w: {k: 0 for k in LAYERS} for w in CANDIDATES}
buoyancy_scores = {w: [] for w in CANDIDATES}

for r in ALL_RUNS:
    for w in CANDIDATES:
        L = first_detection_layer(r['wells'].get(w))
        if L is not None:
            layer_tallies[w][L] += 1
            buoyancy_scores[w].append(LAYER_SCORE_MAP[L])

buoy_mean = {w: float(np.mean(v)) if v else 0.0 for w, v in buoyancy_scores.items()}
buoy_s = pd.Series(buoy_mean)
score3 = (buoy_s - buoy_s.min()) / (buoy_s.max() - buoy_s.min() + 1e-9)
score3 = score3.rename('Score_3_buoyancy_layer')

# Stacked bar: proportion of runs per first-detection layer
n_total = len(ALL_RUNS)
layer_colors = {6:'#1A237E', 7:'#3949AB', 8:'#64B5F6', 9:'#90CAF9', 10:'#BBDEFB'}
fig, ax = plt.subplots(figsize=(9, 5))
bottoms = np.zeros(len(CANDIDATES))
x = np.arange(len(CANDIDATES))
for k in LAYERS:
    vals = np.array([layer_tallies[w][k] / max(n_total, 1) for w in CANDIDATES])
    ax.bar(x, vals, bottom=bottoms, label=f'L{k} first detect', color=layer_colors[k], edgecolor='white')
    bottoms += vals
ax.set_xticks(x); ax.set_xticklabels(CANDIDATES, fontsize=9, rotation=20, ha='right')
ax.set_ylabel('Fraction of runs (first detection layer)', fontsize=11)
ax.set_title('Criterion 3 — Layer of First CO2 Detection (dark=upper=better buoyancy sensitivity)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper right'); ax.grid(True, alpha=0.25, axis='y')
plt.tight_layout(); plt.show()
print(f'Score_3 (buoyancy layer, 0-1, higher=better):\n{score3.round(3).to_string()}')

## Section 5 — Criterion 4: Signal Consistency Across Injection Rates (Robustness)

**Physical rationale:** A surveillance well must be useful across a realistic range of
operating conditions — not only at extreme injection rates.

We count how many of the 7 q_vary runs produce a detection within 1000 yr (`N_detections`).
We also compute the coefficient of variation (CV = std/mean) of `T_detect` for detected runs —
low CV means the well gives a consistent, predictable response regardless of Q.

`Score_4 = 0.7 × N_detections_normalized + 0.3 × (1 − CV_normalized)`

In [ ]:
n_detect = {}
cv_detect = {}
for w in CANDIDATES:
    t_vals = [find_T_detect(r['wells'].get(w)) for r in q_runs]
    detected = [v for v in t_vals if v < 1000]
    n_detect[w] = len(detected)
    if len(detected) > 1:
        cv_detect[w] = float(np.std(detected) / (np.mean(detected) + 1e-9))
    elif len(detected) == 1:
        cv_detect[w] = 0.0
    else:
        cv_detect[w] = 1.0  # worst: no detection

n_s = pd.Series(n_detect, dtype=float)
cv_s = pd.Series(cv_detect, dtype=float)
n_norm  = (n_s  - n_s.min())  / (n_s.max()  - n_s.min()  + 1e-9)
cv_norm = (cv_s - cv_s.min()) / (cv_s.max() - cv_s.min() + 1e-9)
score4 = 0.7 * n_norm + 0.3 * (1 - cv_norm)
score4 = score4.rename('Score_4_consistency')

# Dual-axis bar + line
x = np.arange(len(CANDIDATES))
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
ax1.bar(x - 0.2, [n_detect[w] for w in CANDIDATES], width=0.4,
        color='#2980B9', alpha=0.8, label='N detections (left)', zorder=2)
ax2.plot(x, [cv_detect[w] for w in CANDIDATES], 'o-', color='#E74C3C',
         linewidth=2, markersize=7, label='CV (right)')
ax1.set_xticks(x); ax1.set_xticklabels(CANDIDATES, fontsize=9, rotation=20, ha='right')
ax1.set_ylabel('N detections out of 7 q_vary runs', fontsize=11, color='#2980B9')
ax2.set_ylabel('Coefficient of Variation of T_detect', fontsize=11, color='#E74C3C')
ax1.set_title('Criterion 4 — Detection Consistency Across Injection Rates', fontsize=12, fontweight='bold')
lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lab1+lab2, fontsize=9, loc='upper right')
ax1.grid(True, alpha=0.3, axis='y', zorder=1)
plt.tight_layout(); plt.show()
print(f'Score_4 (consistency, 0-1, higher=better):\n{score4.round(3).to_string()}')

## Section 6 — Criterion 5: Structural Position / Updip Proximity

**Physical rationale:** In the northward-dipping Johansen formation, CO2 migrates updip
toward shallower depths. A shallower well is structurally positioned along the primary
migration pathway and intercepts the plume before it reaches the formation boundary.

Shallower depth (lower absolute depth in metres) = closer to CO2 migration terminus
= structurally better surveillance position.

In [ ]:
depth_s = pd.Series(well_depths).reindex(CANDIDATES)
# Inverse: shallower = higher score
inv_depth = 1.0 / (depth_s + 1e-3)
score5 = (inv_depth - inv_depth.min()) / (inv_depth.max() - inv_depth.min() + 1e-9)
score5 = score5.rename('Score_5_structural_position')

# Horizontal bar sorted by depth
sorted_cands = depth_s.sort_values().index.tolist()
sorted_depths = depth_s[sorted_cands].values
sorted_scores = score5[sorted_cands].values
cmap5 = plt.cm.YlGn
bar_colors = cmap5(sorted_scores)

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(range(len(sorted_cands)), sorted_depths, color=bar_colors, edgecolor='white')
ax.set_yticks(range(len(sorted_cands)))
ax.set_yticklabels(sorted_cands, fontsize=9)
for i, (bar, d) in enumerate(zip(bars, sorted_depths)):
    ax.text(d + 20, i, f'{d:.0f} m', va='center', fontsize=8)
ax.set_xlabel('Mean Reservoir Depth (m below sea level)', fontsize=11)
ax.set_title('Criterion 5 — Structural Position (shallowest at top = most updip)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
sm = plt.cm.ScalarMappable(cmap=cmap5, norm=Normalize(0, 1))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Score_5 (1=shallowest)', shrink=0.8)
plt.tight_layout(); plt.show()
print(f'Score_5 (structural position, 0-1, higher=better):\n{score5.round(3).to_string()}')

## Section 7 — Composite Scoring & Recommendation

All five normalised scores (0–1, higher = better) are combined into a weighted composite.
The weights reflect the relative operational importance of each criterion.

**Output:** Scoring table, radar chart, ranked bar chart, and a plain-text recommendation.

In [ ]:
# Assemble score DataFrame aligned to CANDIDATES
score_df = pd.DataFrame({
    'Score_1 (detection)': score1.reindex(CANDIDATES),
    'Score_2 (pressure)':  score2.reindex(CANDIDATES),
    'Score_3 (buoyancy)':  score3.reindex(CANDIDATES),
    'Score_4 (consistency)': score4.reindex(CANDIDATES),
    'Score_5 (position)':  score5.reindex(CANDIDATES),
}, index=CANDIDATES)

w = WEIGHTS
score_df['Composite_Score'] = (
    score_df['Score_1 (detection)']   * w['detection_speed'] +
    score_df['Score_2 (pressure)']    * w['pressure_signal'] +
    score_df['Score_3 (buoyancy)']    * w['buoyancy_layer'] +
    score_df['Score_4 (consistency)'] * w['consistency'] +
    score_df['Score_5 (position)']    * w['structural_position']
)
score_df = score_df.sort_values('Composite_Score', ascending=False)
best_well   = score_df.index[0]
runner_up   = score_df.index[1]

# Styled table
display(score_df.round(3).style.background_gradient(cmap='Greens',
        subset=['Score_1 (detection)','Score_2 (pressure)','Score_3 (buoyancy)',
                'Score_4 (consistency)','Score_5 (position)','Composite_Score']))

In [ ]:
# ── Radar chart ──────────────────────────────────────────────────
criteria_labels = ['Detection\nSpeed', 'Pressure\nSignal', 'Buoyancy\nLayer',
                   'Consistency', 'Structural\nPosition']
score_cols = ['Score_1 (detection)', 'Score_2 (pressure)', 'Score_3 (buoyancy)',
              'Score_4 (consistency)', 'Score_5 (position)']
N = len(criteria_labels)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_radar = plt.cm.tab10(np.linspace(0, 0.9, len(CANDIDATES)))

for i, well in enumerate(CANDIDATES):
    vals = score_df.loc[well, score_cols].tolist()
    vals += vals[:1]
    comp = score_df.loc[well, 'Composite_Score']
    ax.plot(angles, vals, '-o', color=colors_radar[i], linewidth=2.0,
            label=f'{well} ({comp:.3f})')
    ax.fill(angles, vals, alpha=0.08, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(criteria_labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=7, alpha=0.6)
ax.set_title('Criterion Scores — Radar Chart\n(composite score in legend)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
ax.grid(True, alpha=0.35)
plt.tight_layout(); plt.show()

In [ ]:
# ── Final ranked bar chart ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))
ranked = score_df.sort_values('Composite_Score')
bar_c = ['gold' if w == best_well else '#AAAAAA' for w in ranked.index]
bars = ax.barh(range(len(ranked)), ranked['Composite_Score'], color=bar_c, edgecolor='white')
ax.set_yticks(range(len(ranked)))
ax.set_yticklabels(ranked.index.tolist(), fontsize=10)
for i, (bar, val) in enumerate(zip(bars, ranked['Composite_Score'])):
    ax.text(bar.get_width() + 0.005, i, f'{val:.3f}', va='center', fontsize=9,
            fontweight='bold' if ranked.index[i] == best_well else 'normal')
ax.set_xlabel('Composite Score (0–1)', fontsize=11)
ax.set_title('Surveillance Well Ranking — Composite Score\n(gold = recommended)', fontsize=13, fontweight='bold')
ax.set_xlim(0, ranked['Composite_Score'].max() * 1.2)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

In [ ]:
# ── Plain-text recommendation ─────────────────────────────────────
best_scores = score_df.loc[best_well]
runner_scores = score_df.loc[runner_up]
t_det_mean = tdet_df.loc[best_well, 'T_detect_mean']
dp_val     = dp_mean[best_well]
best_layer_tally = layer_tallies[best_well]
best_first_layer = max(best_layer_tally, key=lambda k: best_layer_tally[k])
det_n = n_detect[best_well]
best_depth = well_depths[best_well]

# depth rank
depth_rank = sorted(CANDIDATES, key=lambda w: well_depths.get(w, 9999)).index(best_well) + 1
t_det_rank = sorted(CANDIDATES, key=lambda w: tdet_df.loc[w,'T_detect_mean']).index(best_well) + 1
dp_rank = sorted(CANDIDATES, key=lambda w: dp_mean.get(w,0), reverse=True).index(best_well) + 1

phys_interp = (
    f"{best_well} sits at {best_depth:.0f} m depth — rank #{depth_rank} shallowest among candidates — "
    f"placing it along the updip CO2 migration pathway in the northward-dipping Johansen formation. "
    f"Its pressure front arrives early (mean detection at {t_det_mean:.0f} yr), and it detects CO2 "
    f"most frequently in layer L{best_first_layer}, consistent with buoyancy-driven plume rise into "
    f"the shallowest permeable intervals. Across all tested injection rates ({det_n}/7 runs produce "
    f"a detection), it provides the most reliable operational signal."
)

bar = '=' * 52
print(f'''
{bar}
  SURVEILLANCE WELL RECOMMENDATION
{bar}
  Recommended well   : {best_well}
  Composite score    : {best_scores['Composite_Score']:.4f}
  Runner-up          : {runner_up} (score: {runner_scores['Composite_Score']:.4f})

  Primary reasons:
    * Earliest mean CO2 detection : {t_det_mean:.0f} yr   (Rank #{t_det_rank})
    * Pressure signal strength    : {dp_val:.2f} bar (Rank #{dp_rank})
    * Most-frequent first detect  : L{best_first_layer} in {best_layer_tally[best_first_layer]}/{len(ALL_RUNS)} runs
    * Detects across {det_n}/7 injection rates tested
    * Structural depth            : {best_depth:.0f} m (Rank #{depth_rank} shallowest)

  Physical interpretation:
    {phys_interp}
{bar}
''')

## Section 8 — Sensitivity Check: Does the Recommendation Hold?

We re-run the composite scoring under four weight schemes:
- **Original** — as configured above
- **(a) Detection-dominated** — 0.6 / 0.1 / 0.1 / 0.1 / 0.1
- **(b) Pressure-dominated** — 0.1 / 0.6 / 0.1 / 0.1 / 0.1
- **(c) Equal weights** — 0.2 each

If the same well wins all four schemes → **robustly optimal**.
If it changes → identify which criterion is the swing factor.

In [ ]:
alt_schemes = {
    'Original': WEIGHTS,
    '(a) Detection-dominated': {'detection_speed':0.60,'pressure_signal':0.10,'buoyancy_layer':0.10,'consistency':0.10,'structural_position':0.10},
    '(b) Pressure-dominated':  {'detection_speed':0.10,'pressure_signal':0.60,'buoyancy_layer':0.10,'consistency':0.10,'structural_position':0.10},
    '(c) Equal weights':       {'detection_speed':0.20,'pressure_signal':0.20,'buoyancy_layer':0.20,'consistency':0.20,'structural_position':0.20},
}

sens_rows = []
for scheme_name, wts in alt_schemes.items():
    comp = (
        score_df['Score_1 (detection)']    * wts['detection_speed'] +
        score_df['Score_2 (pressure)']     * wts['pressure_signal'] +
        score_df['Score_3 (buoyancy)']     * wts['buoyancy_layer'] +
        score_df['Score_4 (consistency)']  * wts['consistency'] +
        score_df['Score_5 (position)']     * wts['structural_position']
    )
    winner = comp.idxmax()
    runner = comp.drop(index=winner).idxmax()
    sens_rows.append({'Weight Scheme': scheme_name,
                      'Recommended': winner,
                      'Score': f'{comp[winner]:.4f}',
                      'Runner-up': runner,
                      'Stable?': '✅ YES' if winner == best_well else f'❌ Changed to {winner}'})

sens_df = pd.DataFrame(sens_rows)
display(sens_df)

all_winners = [r['Recommended'] for r in sens_rows]
if len(set(all_winners)) == 1:
    print(f'\n>>> ROBUSTLY OPTIMAL: {best_well} wins under all {len(all_winners)} weight schemes.')
    print('    The recommendation is insensitive to subjective weight choices.')
else:
    unique_winners = set(all_winners)
    swing = [s for s in alt_schemes if sens_rows[list(alt_schemes.keys()).index(s)]['Recommended'] != best_well]
    print(f'\n>>> SENSITIVE: The recommendation changes under {swing}')
    print(f'    Wells competing: {unique_winners}')
    print('    Review the weights — the swing criterion may require expert elicitation.')

## Summary

This notebook applied five physically motivated criteria to rank six candidate surveillance wells
for the Johansen CO2 storage project:

| Criterion | Weight | Physical basis |
|---|---|---|
| 1. Earliest CO2 detection | 30% | Allows earliest operational response |
| 2. Pressure signal strength | 25% | Pressure fronts arrive before CO2 plume |
| 3. Buoyancy-driven layer detection | 20% | Upper-layer detection = plume tip intercept |
| 4. Consistency across Q | 15% | Operational reliability across rate scenarios |
| 5. Structural depth position | 10% | Updip position = along migration pathway |

The **Section 8 sensitivity check** tests whether the top-ranked well holds its position
when weights are varied — a robustly optimal recommendation requires stability across at
least three of the four schemes.